In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report


data = pd.read_csv(
    "pos_tags.csv",
    nrows=1000
)


data.columns = data.columns.str.lower()


print(data.head())
print(data.columns)



if "word" not in data.columns:

    for col in data.columns:

        if "word" in col:

            data.rename(
                columns={col:"word"},
                inplace=True
            )


if "tag" not in data.columns:

    for col in data.columns:

        if "tag" in col or "pos" in col:

            data.rename(
                columns={col:"tag"},
                inplace=True
            )


print("\nColumns after processing:")
print(data.columns)



sentences = []

temp = []


for _, row in data.iterrows():

    word = row["word"]

    tag = row["tag"]

    temp.append(
        (word,tag)
    )


    if len(temp) == 20:

        sentences.append(temp)

        temp = []



print("\nTotal Sentences:",len(sentences))

print("\nSample Sentence:")
print(sentences[0])



train_data,test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)



vocabulary = set()

tags = set()



for sentence in train_data:

    for word,tag in sentence:

        vocabulary.add(
            word.lower()
        )

        tags.add(tag)



vocabulary = list(vocabulary)

tags = list(tags)



word_to_index = {

    word:i

    for i,word in enumerate(vocabulary)

}



tag_to_index = {

    tag:i

    for i,tag in enumerate(tags)

}



index_to_tag = {

    i:tag

    for tag,i in tag_to_index.items()

}



V = len(vocabulary)

T = len(tags)



print("\nVocabulary Size:",V)

print("Number of Tags:",T)



initial = np.ones(T)

transition = np.ones(
    (T,T)
)

emission = np.ones(
    (T,V)
)



for sentence in train_data:


    first_tag = sentence[0][1]


    initial[
        tag_to_index[first_tag]
    ] += 1



    for i,(word,tag) in enumerate(sentence):


        word = word.lower()


        tag_index = tag_to_index[tag]


        if word in word_to_index:


            emission[
                tag_index,
                word_to_index[word]
            ] += 1



        if i > 0:


            previous_tag = sentence[i-1][1]


            transition[
                tag_to_index[previous_tag],
                tag_index
            ] += 1





initial = initial / initial.sum()



transition = transition / transition.sum(
    axis=1,
    keepdims=True
)



emission = emission / emission.sum(
    axis=1,
    keepdims=True
)




log_initial = np.log(initial)

log_transition = np.log(transition)

log_emission = np.log(emission)




def viterbi(sentence):


    n = len(sentence)


    dp = np.zeros(
        (T,n)
    )


    backpointer = np.zeros(
        (T,n),
        dtype=int
    )



    word = sentence[0].lower()



    if word in word_to_index:


        emit = log_emission[
            :,
            word_to_index[word]
        ]


    else:


        emit = np.log(
            np.ones(T)*1e-10
        )



    dp[:,0] = log_initial + emit




    for i in range(1,n):


        word = sentence[i].lower()



        if word in word_to_index:


            emit = log_emission[
                :,
                word_to_index[word]
            ]


        else:


            emit = np.log(
                np.ones(T)*1e-10
            )



        scores = (

            dp[:,i-1][:,None]

            +

            log_transition

        )



        backpointer[:,i] = np.argmax(
            scores,
            axis=0
        )



        dp[:,i] = (

            np.max(
                scores,
                axis=0
            )

            +

            emit

        )




    best = np.argmax(
        dp[:,-1]
    )


    result = [best]



    for i in range(n-1,0,-1):


        best = backpointer[
            best,
            i
        ]


        result.append(best)



    result.reverse()



    return [

        index_to_tag[i]

        for i in result

    ]




actual = []

predicted = []



for sentence in test_data:


    words = []

    true_tags = []



    for word,tag in sentence:


        words.append(word)

        true_tags.append(tag)



    pred_tags = viterbi(words)



    actual.extend(true_tags)

    predicted.extend(pred_tags)




accuracy = accuracy_score(
    actual,
    predicted
)



print("\nAccuracy:")
print(accuracy)


print(
    classification_report(
        actual,
        predicted,
        zero_division=0
    )
)


test_sentences = [

    "Artificial Intelligence improves healthcare systems",

    "The student reads a book",

    "Machine learning improves prediction",

    "The dog runs quickly",

    "I love natural language processing"

]



print("\nUnseen Sentence Predictions")
print("---------------------------")



for sentence in test_sentences:


    words = sentence.split()


    prediction = viterbi(words)



    print("\nSentence:")
    print(sentence)



    for word,tag in zip(words,prediction):


        print(
            word,
            "---->",
            tag
        )

   unnamed: 0    word pos_tag
0           0      aa      NN
1           1     aaa      NN
2           2     aah      NN
3           3   aahed     VBN
4           4  aahing     VBG
Index(['unnamed: 0', 'word', 'pos_tag'], dtype='object')

Columns after processing:
Index(['unnamed: 0', 'word', 'tag'], dtype='object')

Total Sentences: 50

Sample Sentence:
[('aa', 'NN'), ('aaa', 'NN'), ('aah', 'NN'), ('aahed', 'VBN'), ('aahing', 'VBG'), ('aahs', 'NN'), ('aal', 'NN'), ('aalii', 'NN'), ('aaliis', 'NN'), ('aals', 'NNS'), ('aam', 'NN'), ('aani', 'NN'), ('aardvark', 'NN'), ('aardvarks', 'NNS'), ('aardwolf', 'NN'), ('aardwolves', 'NNS'), ('aargh', 'NN'), ('aaron', 'NN'), ('aaronic', 'NN'), ('aaronical', 'JJ')]

Vocabulary Size: 800
Number of Tags: 9

Accuracy:
0.58
              precision    recall  f1-score   support

          JJ       0.00      0.00      0.00        25
          NN       0.58      1.00      0.73       116
         NNS       0.00      0.00      0.00        29
          RB    